In [ ]:
# Simple setup with compatibility fixes
!pip install "numpy<2.0" "torch<2.6" opencv-python insightface onnxruntime-gpu gradio --force-reinstall
!apt update -qq && apt install -y ffmpeg

import os
os.kill(os.getpid(), 9)

In [ ]:
# Face swap implementation
import cv2
import numpy as np
import insightface
import gradio as gr
import urllib.request

# Download model
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/inswapper_128_fp16.onnx'):
    urllib.request.urlretrieve(
        "https://huggingface.co/hacksider/deep-live-cam/resolve/main/inswapper_128_fp16.onnx",
        "models/inswapper_128_fp16.onnx"
    )

# Initialize models
app = insightface.app.FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))
swapper = insightface.model_zoo.get_model('models/inswapper_128_fp16.onnx', providers=['CUDAExecutionProvider'])

def swap_faces(source_img, target_img, mouth_mask=True):
    if source_img is None or target_img is None:
        return None, "Upload both images"
    
    # Convert RGB to BGR
    source = cv2.cvtColor(source_img, cv2.COLOR_RGB2BGR)
    target = cv2.cvtColor(target_img, cv2.COLOR_RGB2BGR)
    
    # Get faces
    source_faces = app.get(source)
    target_faces = app.get(target)
    
    if not source_faces:
        return target_img, "No face in source"
    if not target_faces:
        return target_img, "No face in target"
    
    # Swap
    result = swapper.get(target, target_faces[0], source_faces[0], paste_back=True)
    
    # Mouth mask
    if mouth_mask and hasattr(target_faces[0], 'landmark_2d_106'):
        landmarks = target_faces[0].landmark_2d_106
        if landmarks is not None and len(landmarks) >= 106:
            mouth_pts = landmarks[[65,66,62,70,69,18,19,20,21,22,23,24,0,8,7,6,5,4,3,2]].astype(int)
            mask = np.zeros(target.shape[:2], dtype=np.uint8)
            cv2.fillPoly(mask, [mouth_pts], 255)
            mask = cv2.GaussianBlur(mask, (15, 15), 5)
            
            x, y, w, h = cv2.boundingRect(mouth_pts)
            x, y = max(0, x-10), max(0, y-10)
            w, h = min(target.shape[1]-x, w+20), min(target.shape[0]-y, h+20)
            
            if w > 0 and h > 0:
                mouth_orig = target[y:y+h, x:x+w]
                mouth_swap = result[y:y+h, x:x+w]
                mask_roi = mask[y:y+h, x:x+w] / 255.0
                
                if mouth_orig.shape == mouth_swap.shape:
                    blended = mouth_orig * mask_roi[:,:,None] + mouth_swap * (1 - mask_roi[:,:,None])
                    result[y:y+h, x:x+w] = blended.astype(np.uint8)
    
    # Convert back to RGB
    result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    return result_rgb, "Success!"

print("Ready!")

In [ ]:
# Launch interface
with gr.Blocks() as demo:
    gr.Markdown("# Face Swap with Mouth Mask")
    
    with gr.Row():
        source = gr.Image(label="Source Face")
        target = gr.Image(label="Target Image")
        result = gr.Image(label="Result")
    
    with gr.Row():
        mouth_check = gr.Checkbox(label="Mouth Mask", value=True)
        btn = gr.Button("Swap", variant="primary")
    
    status = gr.Textbox(label="Status")
    
    btn.click(swap_faces, [source, target, mouth_check], [result, status])

demo.launch(share=True)